# Experiment 5.5.3 — Teacher-weight routing decomposition

Analysis-only notebook. It reads finalized artifacts produced by the Slurm finalizer; it never trains, refits, launches jobs, or regenerates missing runs.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

repo_root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'AGENTS.md').is_file())
root = repo_root / 'notebooks' / 'artifacts' / 'experiment_5_5_3_teacher_weight_routing_decomposition' / 'teacher_weight_routing_decomposition_v1'
required = ['manifest.json', 'teacher_equivalence.csv', 'runs.csv', 'summary.csv', 'paired_deltas.csv', 'recovery.csv', 'logit_distortion.csv', 'weight_drift.csv']
missing = [name for name in required if not (root / name).is_file()]
if missing:
    raise FileNotFoundError(f'Missing finalized Exp5.5.3 artifacts: {missing}')
manifest = json.loads((root / 'manifest.json').read_text(encoding='utf-8'))
equiv = pd.read_csv(root / 'teacher_equivalence.csv')
runs = pd.read_csv(root / 'runs.csv')
summary = pd.read_csv(root / 'summary.csv')
paired = pd.read_csv(root / 'paired_deltas.csv')
recovery = pd.read_csv(root / 'recovery.csv')
distortion = pd.read_csv(root / 'logit_distortion.csv')
weight_drift = pd.read_csv(root / 'weight_drift.csv')
display(pd.DataFrame([manifest]))
display(equiv)
display(summary)

## Figure 1 — Complete balanced-accuracy ladder

In [ ]:
order = ['hard_teacher_frozen', 'soft_teacher_frozen', 'hard_random_train', 'soft_random_train', 'soft_teacher_init_retrain', 'soft_linear_refit']
fig, ax = plt.subplots(figsize=(11, 5.5))
x = np.arange(len(order))
width = 0.36
for offset, family in zip((-width/2, width/2), ('relative10', 'fixed250')):
    values = summary.set_index(['family', 'condition']).loc[[(family, c) for c in order], 'mean_test_balanced_accuracy'].to_numpy()
    errors = summary.set_index(['family', 'condition']).loc[[(family, c) for c in order], 'sem_test_balanced_accuracy'].to_numpy()
    ax.bar(x + offset, values, width=width, yerr=errors, capsize=3, label=family)
ax.set_xticks(x, order, rotation=25, ha='right')
ax.set_ylabel('Mean test balanced accuracy')
ax.set_title('Exp5.5.3: routing / weight-learning decomposition')
ax.legend()
ax.grid(axis='y', alpha=0.25)
plt.tight_layout()
plt.show()

## Figure 2 — Paired hard-teacher → soft-teacher change

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 5.0))
for family in ('relative10', 'fixed250'):
    subset = runs[(runs.family == family) & runs.condition.isin(['hard_teacher_frozen', 'soft_teacher_frozen'])]
    pivot = subset.pivot(index='seed', columns='condition', values='test_balanced_accuracy')
    for seed, row in pivot.iterrows():
        ax.plot([0, 1], [row['hard_teacher_frozen'], row['soft_teacher_frozen']], marker='o', alpha=0.6)
ax.set_xticks([0, 1], ['Hard teacher', 'Soft teacher'])
ax.set_ylabel('Test balanced accuracy')
ax.set_title('Paired hard → soft teacher routing')
ax.grid(axis='y', alpha=0.25)
plt.tight_layout()
plt.show()
display(paired[paired.comparison == 'softness_penalty'])

## Figure 3 — Teacher-init retraining recovery

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 5.0))
for family in ('relative10', 'fixed250'):
    part = recovery[recovery.family == family].sort_values('seed')
    ax.plot(part.seed, part.retraining_recovery_fraction, marker='o', label=family)
ax.axhline(1.0, linestyle='--', linewidth=1)
ax.axhline(0.0, linestyle=':', linewidth=1)
ax.set_xlabel('Seed')
ax.set_ylabel('Recovery fraction')
ax.set_title('How much hard→soft teacher loss is recovered by teacher-init retraining?')
ax.legend()
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()
display(recovery)

## Figure 4 — Optimization controls

In [ ]:
controls = paired[paired.comparison.isin(['hard_optimization_gap', 'soft_linear_minus_soft_random'])].copy()
fig, ax = plt.subplots(figsize=(8.0, 5.0))
means = controls.groupby(['family', 'comparison']).delta_test_balanced_accuracy.mean().unstack()
means.plot(kind='bar', ax=ax)
ax.axhline(0.0, linewidth=1)
ax.set_ylabel('Paired test BA delta')
ax.set_title('Optimization gaps: neural training vs proven/linear solutions')
ax.set_xlabel('')
ax.legend(title='comparison')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()
display(controls)

## Figure 5 — Frozen hard→soft logit distortion

In [ ]:
test_dist = distortion[distortion.split == 'test'].copy()
fig, ax = plt.subplots(figsize=(8.0, 5.0))
for family in ('relative10', 'fixed250'):
    part = test_dist[test_dist.family == family].sort_values('seed')
    ax.scatter(part.mean_logit_l2, part.mean_true_margin_change, label=family, s=55)
    for _, row in part.iterrows():
        ax.annotate(str(int(row.seed)), (row.mean_logit_l2, row.mean_true_margin_change), xytext=(4, 4), textcoords='offset points', fontsize=8)
ax.axhline(0.0, linewidth=1)
ax.set_xlabel('Mean hard→soft logit L2 distortion')
ax.set_ylabel('Mean true-class margin change')
ax.set_title('Frozen-teacher routing distortion')
ax.legend()
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()
display(test_dist)
display(weight_drift)